In [14]:
%pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [ollama] 7/10 [anyio]ic]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
import json
import ollama  # pip install ollama
import re
from pathlib import Path
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple
import logging

# Configuration
model_name = "llama3.1:70b"
input_file = "/data/gregIB/issuebench/2_final_dataset/combined_prompts_issues_with_topics.csv"

safe_model_name = re.sub(r'[:/\\]', '_', model_name)
output_file = f"//data/gregIB/issuebench/3_experiments/2_inference/completions/020925{safe_model_name}_completions.csv"

# Processing parameters
TEST_SUBSET = 5   # e.g. 200 to test first 200 todos; None = all
MAX_WORKERS = 8                  
REQUEST_TIMEOUT = 120            
RETRY_ATTEMPTS = 2               
RETRY_BACKOFF_SECS = 2           
BATCH_SIZE = 24                 

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
log = logging.getLogger("ollama-run")

def query_ollama_local(model: str, prompt: str) -> str:
    """Call Ollama locally using the ollama Python library."""
    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            stream=False,
            options={
                'temperature': 0.7,
                'top_p': 0.9,
                'num_predict': -1,
            }
        )
        return response['response'].strip()
    except Exception as e:
        raise RuntimeError(f"Ollama local call failed: {e}")

def complete_with_retries(prompt: str) -> str:
    """Retry wrapper around query_ollama_local with simple backoff."""
    last_err = None
    for attempt in range(RETRY_ATTEMPTS + 1):
        try:
            return query_ollama_local(model_name, prompt)
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS:
                sleep_for = RETRY_BACKOFF_SECS * (attempt + 1)
                log.warning(f"Request failed (attempt {attempt+1}/{RETRY_ATTEMPTS+1}). "
                            f"Retrying in {sleep_for:.1f}s... | Error: {e}")
                time.sleep(sleep_for)
            else:
                raise last_err

def list_available_models():
    """List all locally available models."""
    try:
        models = ollama.list()
        log.info("Available models:")
        for model in models['models']:
            log.info(f"  - {model['name']} ({model.get('size', 'unknown')} bytes)")
        return [model['name'] for model in models['models']]
    except Exception as e:
        log.error(f"Error listing models: {e}")
        return []

def ensure_model_available(model: str):
    """Check if model is available locally, pull if needed."""
    try:
        available_models = list_available_models()
        if model not in available_models:
            log.warning(f"Model {model} not found locally.")
            log.info(f"Attempting to pull model {model}...")
            ollama.pull(model)
            log.info(f"Successfully pulled model {model}")
        else:
            log.info(f"Model {model} is available locally")
    except Exception as e:
        log.error(f"Error checking/pulling model: {e}")
        raise

def process_row(i: int, df: pd.DataFrame) -> tuple[int, str]:
    """Process a single row."""
    prompt = str(df.at[i, "prompt_text"])
    resp = complete_with_retries(prompt)
    return i, resp

def main():
    """Main processing function."""
    # Load data and prepare for processing
    df = pd.read_csv(input_file)
    if "model" not in df.columns:
        df["model"] = ""

    mask_todo = df["response_text"].isna() | (df["response_text"].astype(str).str.strip() == "")
    todo_idx = df.index[mask_todo].tolist()
    if TEST_SUBSET is not None:
        todo_idx = todo_idx[:TEST_SUBSET]

    log.info(f"Rows total = {len(df)} | to-complete = {len(todo_idx)} | model = {model_name}")
    log.info(f"Output file: {output_file}")

    # Ensure model is available
    ensure_model_available(model_name)

    processed = 0
    start_time = time.time()

    # Process in batches with threading
    for start in range(0, len(todo_idx), BATCH_SIZE):
        batch = todo_idx[start:start+BATCH_SIZE]
        futures = []
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for i in batch:
                futures.append(ex.submit(process_row, i, df))
            
            for fut in as_completed(futures):
                try:
                    i, resp = fut.result()
                    df.at[i, "response_text"] = resp
                    df.at[i, "model"] = model_name
                    processed += 1
                except Exception as e:
                    log.error(f"Failed to process row: {e}")
                    continue

        # Checkpoint save
        df.to_csv(output_file, index=False)
        elapsed = time.time() - start_time
        rate = processed / elapsed if elapsed > 0 else 0
        log.info(f"Checkpoint: {processed}/{len(todo_idx)} completed ({rate:.2f} req/s) -> {output_file}")

    elapsed = time.time() - start_time
    log.info(f"Done. Wrote: {output_file} | processed {processed} rows in {elapsed:.1f}s")
    if processed > 0:
        log.info(f"Average rate: {processed/elapsed:.2f} requests/second")

if __name__ == "__main__":
    main()

2025-09-02 14:53:58,513 | INFO | Rows total = 62178 | to-complete = 100 | model = llama3.1:70b
2025-09-02 14:53:58,513 | INFO | Output file: //data/gregIB/issuebench/3_experiments/2_inference/completions/020925llama3.1_70b_completions.csv
2025-09-02 14:53:58,517 | INFO | HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"
2025-09-02 14:53:58,519 | INFO | Available models:
2025-09-02 14:53:58,519 | ERROR | Error listing models: 'name'
2025-09-02 14:53:58,520 | WARNING | Model llama3.1:70b not found locally.
2025-09-02 14:53:58,521 | INFO | Attempting to pull model llama3.1:70b...
2025-09-02 14:53:59,311 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/pull "HTTP/1.1 200 OK"
2025-09-02 14:53:59,312 | INFO | Successfully pulled model llama3.1:70b
2025-09-02 14:54:37,012 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
/tmp/ipykernel_6809/371677982.py:133: FutureWarning: Setting an item of incompatible dtype is deprecated and will rais

KeyboardInterrupt: 

2025-09-02 14:57:20,830 | INFO | HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
